In [ ]:
k_alvo = 2
l_alvo = 2
t_alvo = 0.20

# CONFIG
usar_k = False
usar_l = False
usar_t = True

colunas_sensiveis_l = ['cholesterol', 'gluc']
coluna_sensivel_t = 'cardio'

In [ ]:
!pip install faker -q

import pandas as pd
import numpy as np
from faker import Faker
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
import warnings
import os

In [44]:
arquivo_original = 'cardio_train.csv'
arquivo_personal = 'personal_cardio_train.csv'

if os.path.exists(arquivo_original):
    caminho_original = arquivo_original
    caminho_personal = arquivo_personal
else:
    diretorio_atual = os.getcwd()
    
    if os.path.basename(diretorio_atual) == 'src':
        caminho_original = os.path.join(diretorio_atual, '..', 'data', arquivo_original)
        caminho_personal = os.path.join(diretorio_atual, '..', 'data', arquivo_personal)
    else:
        caminho_original = os.path.join(diretorio_atual, 'data', arquivo_original)
        caminho_personal = os.path.join(diretorio_atual, 'data', arquivo_personal)

caminho_original = os.path.abspath(caminho_original)
caminho_personal = os.path.abspath(caminho_personal)

arquivo_anonimizado = 'anonimized_cardio_train.csv'

if os.path.exists('cardio_train.csv'):
    caminho_anonimizado = arquivo_anonimizado
else:
    diretorio_atual = os.getcwd()
    if os.path.basename(diretorio_atual) == 'src':
        caminho_anonimizado = os.path.join(diretorio_atual, '..', 'data', arquivo_anonimizado)
    else:
        caminho_anonimizado = os.path.join(diretorio_atual, 'data', arquivo_anonimizado)

caminho_anonimizado = os.path.abspath(caminho_anonimizado)

In [45]:
try:
    df_real = pd.read_csv(caminho_original, sep=';')
        
    print(f"Dataset carregado: {df_real.shape[0]} linhas e {df_real.shape[1]} colunas.")
    

    if 'age' in df_real.columns:
        df_real['age'] = (df_real['age'] / 365.25).astype(int)
        print("Transformação: Coluna 'age' convertida de dias para anos.")
    

    fake = Faker('pt_BR')
    print("A injetar Nomes e CPFs fictícios (Mock de Dados Pessoais)")
    
    if 'nome' not in df_real.columns:
        nomes_por_genero = []
        for g in df_real['gender']:
            if g == 1:
                nomes_por_genero.append(fake.name_female())
            elif g == 2:
                nomes_por_genero.append(fake.name_male())
            else:
                nomes_por_genero.append(fake.name())
                
        df_real.insert(0, 'nome', nomes_por_genero)
    
    if 'cpf' not in df_real.columns:
        df_real.insert(1, 'cpf', [fake.ssn() for i in range(len(df_real))])


    if os.path.dirname(caminho_personal):
        os.makedirs(os.path.dirname(caminho_personal), exist_ok=True)
    df_real.to_csv(caminho_personal, index=False, sep=';')
    
    print(f"Arquivo processado e salvo em: {caminho_personal}\n")
    display(df_real[['nome', 'cpf', 'id', 'age', 'gender', 'height',
                     'weight', 'ap_hi', 'ap_lo', 'cholesterol', 'gluc',
                     'smoke', 'alco', 'cardio']].head(3))

except FileNotFoundError:
    print(f"O arquivo {caminho_original} não foi encontrado.")

Dataset carregado: 70000 linhas e 13 colunas.
Transformação: Coluna 'age' convertida de dias para anos.
A injetar Nomes e CPFs fictícios (Mock de Dados Pessoais)
Arquivo processado e salvo em: /home/usuario/Academic/2026.1/TCC1/TCC-PoC1-Random-Forest-com-Dados-Anonimizados/data/personal_cardio_train.csv



,nome,cpf,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,cardio
0,Sr. Raul da Costa,43065819260,0,50,2,168,62.0,110,80,1,1,0,0,0
1,Elisa Costela,29817405630,1,55,1,156,85.0,140,90,3,1,0,0,1
2,Eduarda Camargo,39564810701,2,51,1,165,64.0,130,70,3,1,0,0,1


In [ ]:
df_pessoal = pd.read_csv(caminho_personal, sep=';')
df_anon = df_pessoal.copy()

# Supressão de identificadores diretos e generalização de QIs
df_anon = df_anon.drop(columns=['nome', 'cpf', 'id'])

df_anon['age_group']    = pd.cut(df_anon['age'],    bins=[0, 35, 50, 65, 120],      labels=['Jovem', 'Adulto', 'Meia-Idade', 'Sênior'])
df_anon['weight_group'] = pd.cut(df_anon['weight'], bins=[0, 50, 65, 80, 95, 200],  labels=['<50kg', '50-65kg', '66-80kg', '81-95kg', '>95kg'])
df_anon['height_group'] = pd.cut(df_anon['height'], bins=[0, 150, 160, 170, 180, 250], labels=['<150cm', '150-160cm', '161-170cm', '171-180cm', '>180cm'])

df_anon = df_anon.drop(columns=['age', 'weight', 'height'])

quase_identificadores_finais = ['age_group', 'gender', 'weight_group', 'height_group']

df_final = df_anon.copy()

if usar_k:
    print(f"Aplicar K-Anonimato (k={k_alvo}).")

    contagem = df_anon.groupby(quase_identificadores_finais, observed=False).size().reset_index(name='Tamanho')
    df_k_anon = df_anon.merge(contagem, on=quase_identificadores_finais)

    df_final = df_k_anon[df_k_anon['Tamanho'] >= k_alvo].copy()
    df_final = df_final.drop(columns=['Tamanho'])

    registros_perdidos_k = len(df_anon) - len(df_final)
    print(f"-> Registros mantidos após K-Anonimato: {len(df_final)}")
    print(f"-> Registros suprimidos pelo K-Anonimato: {registros_perdidos_k}")
else:
    print("K-Anonimato desativado: ignorado.")

df_final.to_csv(caminho_anonimizado, index=False, sep=';')
print(f"Arquivo salvo em: {caminho_anonimizado}\n")
display(df_final.head(3))


K-Anonimato desativado — ignorado.
Arquivo salvo em: /home/usuario/Academic/2026.1/TCC1/TCC-PoC1-Random-Forest-com-Dados-Anonimizados/data/anonimized_cardio_train.csv



,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_group,weight_group,height_group
0,2,110,80,1,1,0,0,1,0,Adulto,50-65kg,161-170cm
1,1,140,90,3,1,0,0,1,1,Meia-Idade,81-95kg,150-160cm
2,1,130,70,3,1,0,0,0,1,Meia-Idade,50-65kg,161-170cm


In [ ]:
if usar_l:
    print(f"Aplicar L-Diversidade (l={l_alvo}) nos atributos sensíveis: {colunas_sensiveis_l}.")

    df_l_diversidade = df_final.copy()

    df_l_diversidade['comb_sensivel'] = df_l_diversidade[colunas_sensiveis_l].astype(str).agg('_'.join, axis=1)

    # Agrupar pelos quase-identificadores e contar as combinações ÚNICAS da coluna composta
    diversidade = df_l_diversidade.groupby(quase_identificadores_finais, observed=False)['comb_sensivel'].nunique().reset_index(name='l_valores_distintos')

    # Mesclar com o dataset da etapa anterior
    df_l_div = df_l_diversidade.merge(diversidade, on=quase_identificadores_finais)

    # Filtrar para manter APENAS os grupos que atendem à regra de l-diversidade
    df_l_diversidade = df_l_div[df_l_div['l_valores_distintos'] >= l_alvo].copy()
    df_l_diversidade = df_l_diversidade.drop(columns=['comb_sensivel', 'l_valores_distintos'])

    registros_perdidos = len(df_final) - len(df_l_diversidade)
    print(f"-> Registros mantidos após L-Diversidade: {len(df_l_diversidade)}")
    print(f"-> Registros suprimidos pela L-Diversidade: {registros_perdidos}")

    df_l_diversidade.to_csv(caminho_anonimizado, index=False, sep=';')
    df_final = df_l_diversidade
    display(df_final.head(3))
else:
    print("L-Diversidade desativada: ignorada.")


L-Diversidade desativada — ignorada.


In [ ]:
if usar_t:
    print(f"Aplicar T-Proximidade (t={t_alvo}) no atributo sensível: '{coluna_sensivel_t}'.")
    print("Métrica: Distância de Variação Total (TV distance) - adequada para atributo binário.")

    df_t_proximidade = df_final.copy()

    prob_global = df_t_proximidade[coluna_sensivel_t].value_counts(normalize=True).to_dict()

    grupos = df_t_proximidade.groupby(quase_identificadores_finais, observed=False)
    grupos_validos = []

    for nome_grupo, grupo in grupos:
        if len(grupo) == 0:
            continue

        prob_local = grupo[coluna_sensivel_t].value_counts(normalize=True).to_dict()

        # TV distance: 0.5 * Σ|p_local(c) - p_global(c)| para cada categoria c
        categorias = set(prob_global.keys()).union(set(prob_local.keys()))
        distancia = 0.5 * sum(abs(prob_local.get(c, 0) - prob_global.get(c, 0)) for c in categorias)

        if distancia <= t_alvo:
            grupos_validos.append(nome_grupo)

    chaves_grupos = df_t_proximidade.set_index(quase_identificadores_finais).index
    df_t_proximidade = df_t_proximidade[chaves_grupos.isin(grupos_validos)].copy()

    registros_perdidos_t = len(df_final) - len(df_t_proximidade)
    print(f"-> Registros mantidos após T-Proximidade: {len(df_t_proximidade)}")
    print(f"-> Registros suprimidos pela T-Proximidade: {registros_perdidos_t}")

    df_t_proximidade.to_csv(caminho_anonimizado, index=False, sep=';')
    df_final = df_t_proximidade
    display(df_final.head(3))
else:
    print("T-Proximidade desativada: ignorada.")


Aplicar T-Proximidade (t=0.2) no atributo sensível: 'cardio'.
Métrica: Distância de Variação Total (TV distance) — adequada para atributo binário.
-> Registros mantidos após T-Proximidade: 58606
-> Registros suprimidos pela T-Proximidade: 11394


,gender,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_group,weight_group,height_group
1,1,140,90,3,1,0,0,1,1,Meia-Idade,81-95kg,150-160cm
2,1,130,70,3,1,0,0,0,1,Meia-Idade,50-65kg,161-170cm
3,2,150,100,1,1,0,0,1,1,Adulto,81-95kg,161-170cm


In [49]:
coluna_alvo = 'cardio'
colunas_medicas = ['ap_hi', 'ap_lo', 'cholesterol', 'gluc', 'smoke', 'alco', 'active']

def treinar_modelo(df_treino, features, alvo):
    X = df_treino[features]
    y = df_treino[alvo]

    cols_categoricas = X.select_dtypes(include=['object', 'category']).columns.tolist()
    preprocessor = ColumnTransformer(
        transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), cols_categoricas)],
        remainder='passthrough'
    )

    X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

    modelo = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10))
    ])

    modelo.fit(X_treino, y_treino)
    return accuracy_score(y_teste, modelo.predict(X_teste))

print("[1/2] Treinar IA sem anonimizar.")
features_baseline = ['age', 'gender', 'height', 'weight'] + colunas_medicas
acuracia_baseline = treinar_modelo(df_pessoal, features_baseline, coluna_alvo)

print("[2/2] Treinar IA com dados anonimizados.")
features_anon = quase_identificadores_finais + colunas_medicas
acuracia_anon = treinar_modelo(df_final, features_anon, coluna_alvo)

# Label dinâmico com as técnicas ativas
tecnicas_ativas = []
if usar_k: tecnicas_ativas.append(f"K={k_alvo}")
if usar_l: tecnicas_ativas.append(f"L={l_alvo}")
if usar_t: tecnicas_ativas.append(f"T={t_alvo}")
label_tecnicas = " + ".join(tecnicas_ativas) if tecnicas_ativas else "Nenhuma técnica ativa"

print("\n" + "="*60)
print(" PoC1 - RANDOM-FOREST COM DADOS ANONIMIZADOS")
print("="*60)
print(f"Técnicas aplicadas: {label_tecnicas}")
print(f"Total de Registros (Baseline):    {len(df_pessoal)}")
print(f"Registros Mantidos (Anonimizado): {len(df_final)}")
print(f"Registros Suprimidos:             {len(df_pessoal) - len(df_final)} ({100.0 - (len(df_final) * 100) / len(df_pessoal):.2f}%)")
print("-" * 60)
print(f"Acurácia Baseline (Sem Privacidade):   {acuracia_baseline*100:.2f}%")
print(f"Acurácia Anonimizada ({label_tecnicas}): {acuracia_anon*100:.2f}%")
print(f"Custo de Privacidade (Queda):          {(acuracia_baseline - acuracia_anon)*100:.2f} pontos percentuais")
print("="*60)


[1/2] Treinar IA sem anonimizar.
[2/2] Treinar IA com dados anonimizados.

 PoC1 - RANDOM-FOREST COM DADOS ANONIMIZADOS
Técnicas aplicadas: T=0.2
Total de Registros (Baseline):    70000
Registros Mantidos (Anonimizado): 58606
Registros Suprimidos:             11394 (16.28%)
------------------------------------------------------------
Acurácia Baseline (Sem Privacidade):   74.01%
Acurácia Anonimizada (T=0.2): 71.97%
Custo de Privacidade (Queda):          2.04 pontos percentuais


In [ ]:
tecnicas = []
if usar_k: tecnicas.append("K")
if usar_l: tecnicas.append("L")
if usar_t: tecnicas.append("T")
cenario_nome = "+".join(tecnicas) if tecnicas else "Nenhuma"

df_resultado = pd.DataFrame({
    'cenario': [cenario_nome],
    'usar_k': [usar_k],
    'usar_l': [usar_l],
    'usar_t': [usar_t],
    'k_alvo': [k_alvo if usar_k else None],
    'l_alvo': [l_alvo if usar_l else None],
    't_alvo': [t_alvo if usar_t else None],
    'total_registros': [len(df_pessoal)],
    'registros_mantidos': [len(df_final)],
    'registros_suprimidos': [len(df_pessoal) - len(df_final)],
    'registros_suprimidos_porcentagem': [100.0 - (len(df_final) * 100) / len(df_pessoal)],
    'acuracia_sem_privacidade': [acuracia_baseline * 100],
    'acuracia_com_privacidade': [acuracia_anon * 100],
    'queda_de_privacidade_porcentagem': [(acuracia_baseline - acuracia_anon) * 100]
})

os.makedirs("outputs", exist_ok=True)

nome_arquivo = f"parcial_k{k_alvo}_l{l_alvo}_t{t_alvo}_uk{usar_k}_ul{usar_l}_ut{usar_t}.csv"
df_resultado.to_csv(f"outputs/{nome_arquivo}", index=False)